# 02 - Feature Engineering with Snowflake Feature Store

## Anomaly Detection for Reconciliation Variances

This notebook uses **Snowflake Feature Store** to:
- Register entities and feature views
- Create managed feature views with automatic refresh
- Generate point-in-time correct training data
- Prepare features for ML model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, lag, avg, stddev, max as sf_max, count, when, lit, abs as sf_abs, sum as sf_sum, count_distinct
from snowflake.snowpark.window import Window
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from sklearn.preprocessing import StandardScaler

session = get_active_session()

FEATURE_COLUMNS = [
    'VARIANCE_AMOUNT', 'GL_BANK_DIFF', 'GL_SUBLEDGER_DIFF', 'RECON_COUNT', 'UNIDENTIFIED_AMOUNT',
    'VARIANCE_PCT_CHANGE', 'VARIANCE_Z_SCORE', 'VARIANCE_Z_SCORE_PERIOD', 'VARIANCE_PCT_OF_BALANCE',
    'VARIANCE_VS_ROLLING_MAX', 'IS_ABOVE_P95', 'GL_BANK_DIFF_RATIO', 'BALANCE_CHANGE_PCT',
    'HIERARCHY_DEPTH_NORMALIZED', 'IS_KEY_ACCOUNT_FLAG', 'ROLLING_AVG_VARIANCE_3', 'ROLLING_STD_VARIANCE_3',
    'ROLLING_MAX_VARIANCE_6', 'ENTITY_AVG_VARIANCE', 'ENTITY_ASSIGNMENT_COUNT'
]

print(f"Connected to Snowflake")
print(f"Features to engineer: {len(FEATURE_COLUMNS)}")

## 1. Configuration

Set your database, schema, and warehouse for the Feature Store.

In [ ]:
DATABASE = "COCO_LIVE_DB"
SCHEMA = "DBT"
WAREHOUSE = "SNOW_INTELLIGENCE_DEMO_WH"
SOURCE_TABLE = f"{DATABASE}.{SCHEMA}.RECONCILIATION_360"

print(f"Feature Store: {DATABASE}.{SCHEMA}")
print(f"Warehouse: {WAREHOUSE}")
print(f"Source Table: {SOURCE_TABLE}")

## 2. Initialize Feature Store

Create a connection to the Snowflake Feature Store. This creates the necessary infrastructure if it doesn't exist.

In [ ]:
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print(f"Feature Store initialized: {DATABASE}.{SCHEMA}")

## 3. Register Entity

Entities define the join keys for feature lookups. For reconciliation data, we use assignment + period as the compound key.

In [ ]:
entity = Entity(
    name="RECONCILIATION_ASSIGNMENT",
    join_keys=["ASSIGNMENT_ID", "PERIOD_ID"],
    desc="Unique reconciliation assignment for each period"
)
fs.register_entity(entity)
print(f"Entity registered: {entity.name}")
print(f"Join keys: {entity.join_keys}")

## 4. Create Feature View

Create a managed feature view with automatic refresh. This generates 20 engineered features:
- **Temporal features**: Rolling statistics, period-over-period changes
- **Statistical features**: Z-scores, percentile rankings
- **Entity aggregations**: Cross-assignment patterns

In [ ]:
def build_feature_dataframe(session, source_table):
    """Build feature transformations using Snowpark."""
    base = session.table(source_table).filter(col("IS_ACTIVE") == True)
    
    assignment_window = Window.partition_by("ASSIGNMENT_ID").order_by("PERIOD_END_DATE")
    entity_window = Window.partition_by("ENTITY_ID")
    rolling_window_3 = Window.partition_by("ASSIGNMENT_ID").order_by("PERIOD_END_DATE").rows_between(-3, -1)
    rolling_window_6 = Window.partition_by("ASSIGNMENT_ID").order_by("PERIOD_END_DATE").rows_between(-6, -1)
    
    features_df = base.select(
        col("ASSIGNMENT_ID"), col("PERIOD_ID"), col("PERIOD_END_DATE"),
        col("ENTITY_ID"), col("ENTITY_NAME"), col("ACCOUNT_COMBINATION"),
        col("RECONCILIATION_STATUS"), col("IS_KEY_ACCOUNT"), col("HIERARCHY_DEPTH"),
        col("BALANCE_GL"),
        col("TOTAL_ABS_VARIANCE").alias("VARIANCE_AMOUNT"),
        col("GL_BANK_DIFFERENCE").alias("GL_BANK_DIFF"),
        col("GL_SUBLEDGER_DIFFERENCE").alias("GL_SUBLEDGER_DIFF"),
        col("RECONCILIATION_COUNT").alias("RECON_COUNT"),
        col("TOTAL_UNIDENTIFIED_AMOUNT").alias("UNIDENTIFIED_AMOUNT")
    )
    
    features_df = features_df.with_column("PREV_VARIANCE", lag("VARIANCE_AMOUNT", 1).over(assignment_window))
    features_df = features_df.with_column("PREV_BALANCE_GL", lag("BALANCE_GL", 1).over(assignment_window))
    features_df = features_df.with_column("ROLLING_AVG_VARIANCE_3", avg("VARIANCE_AMOUNT").over(rolling_window_3))
    features_df = features_df.with_column("ROLLING_STD_VARIANCE_3", stddev("VARIANCE_AMOUNT").over(rolling_window_3))
    features_df = features_df.with_column("ROLLING_MAX_VARIANCE_6", sf_max("VARIANCE_AMOUNT").over(rolling_window_6))
    features_df = features_df.with_column("ENTITY_AVG_VARIANCE", avg("VARIANCE_AMOUNT").over(entity_window))
    features_df = features_df.with_column("ENTITY_ASSIGNMENT_COUNT", count("*").over(entity_window))
    
    features_df = features_df.with_column("VARIANCE_PCT_CHANGE",
        when(col("PREV_VARIANCE").is_null() | (col("PREV_VARIANCE") == 0), lit(0.0))
        .otherwise((col("VARIANCE_AMOUNT") - col("PREV_VARIANCE")) / col("PREV_VARIANCE")))
    
    features_df = features_df.with_column("VARIANCE_Z_SCORE",
        when(col("ROLLING_STD_VARIANCE_3").is_null() | (col("ROLLING_STD_VARIANCE_3") == 0), lit(0.0))
        .otherwise((col("VARIANCE_AMOUNT") - col("ROLLING_AVG_VARIANCE_3")) / col("ROLLING_STD_VARIANCE_3")))
    
    features_df = features_df.with_column("VARIANCE_Z_SCORE_PERIOD", lit(0.0))
    
    features_df = features_df.with_column("VARIANCE_PCT_OF_BALANCE",
        when(col("BALANCE_GL") == 0, lit(0.0))
        .otherwise(col("VARIANCE_AMOUNT") / sf_abs(col("BALANCE_GL"))))
    
    features_df = features_df.with_column("VARIANCE_VS_ROLLING_MAX",
        when(col("ROLLING_MAX_VARIANCE_6").is_null() | (col("ROLLING_MAX_VARIANCE_6") == 0), lit(0.0))
        .otherwise(col("VARIANCE_AMOUNT") / col("ROLLING_MAX_VARIANCE_6")))
    
    features_df = features_df.with_column("IS_ABOVE_P95", lit(0))
    
    features_df = features_df.with_column("GL_BANK_DIFF_RATIO",
        when(col("BALANCE_GL") == 0, lit(0.0))
        .otherwise(col("GL_BANK_DIFF") / sf_abs(col("BALANCE_GL"))))
    
    features_df = features_df.with_column("BALANCE_CHANGE_PCT",
        when(col("PREV_BALANCE_GL").is_null() | (col("PREV_BALANCE_GL") == 0), lit(0.0))
        .otherwise((col("BALANCE_GL") - col("PREV_BALANCE_GL")) / sf_abs(col("PREV_BALANCE_GL"))))
    
    features_df = features_df.with_column("HIERARCHY_DEPTH_NORMALIZED", col("HIERARCHY_DEPTH") / lit(10.0))
    
    features_df = features_df.with_column("IS_KEY_ACCOUNT_FLAG",
        when(col("IS_KEY_ACCOUNT") == True, lit(1)).otherwise(lit(0)))
    
    features_df = features_df.with_column("IS_ANOMALY_LABEL",
        when((col("RECONCILIATION_STATUS") == "High Variance") | 
             (col("VARIANCE_Z_SCORE") > 3) | 
             (col("VARIANCE_Z_SCORE") < -3), lit(1))
        .otherwise(lit(0)))
    
    final_columns = ["ASSIGNMENT_ID", "PERIOD_ID", "PERIOD_END_DATE", "ENTITY_ID", "ENTITY_NAME",
                     "ACCOUNT_COMBINATION", "RECONCILIATION_STATUS"] + FEATURE_COLUMNS + ["IS_ANOMALY_LABEL"]
    
    return features_df.select(*final_columns)

feature_df = build_feature_dataframe(session, SOURCE_TABLE)
print(f"Feature DataFrame built with {len(FEATURE_COLUMNS)} features")

In [ ]:
feature_df

## 5. Register Feature View

Register the feature view in the Feature Store with versioning.

In [ ]:
fv = FeatureView(
    name="ANOMALY_DETECTION_FEATURES",
    entities=[entity],
    feature_df=feature_df,
    timestamp_col="PERIOD_END_DATE",
    refresh_freq="1 hour",
    desc="Engineered features for anomaly detection on reconciliation variances"
)

registered_fv = fs.register_feature_view(
    feature_view=fv,
    version="v1",
    block=True,
    overwrite=True
)

print(f"Feature View registered: {registered_fv.name} v1")
print(f"Status: {registered_fv.status}")

## 6. List Registered Feature Views

In [ ]:
feature_views_df = fs.list_feature_views().to_pandas()
print("Registered Feature Views:")
for _, row in feature_views_df.iterrows():
    print(f"  - {row['NAME']} ({row['VERSION']})")
feature_views_df

## 7. Validate Feature Data

In [ ]:
stats_df = fs.read_feature_view(registered_fv).select(
    count("*").alias("TOTAL_RECORDS"),
    count_distinct("ASSIGNMENT_ID").alias("UNIQUE_ASSIGNMENTS"),
    count_distinct("PERIOD_ID").alias("UNIQUE_PERIODS"),
    sf_sum("IS_ANOMALY_LABEL").alias("LABELED_ANOMALIES")
).to_pandas()

stats_df["ANOMALY_RATE_PCT"] = round(100.0 * stats_df["LABELED_ANOMALIES"] / stats_df["TOTAL_RECORDS"], 2)
print("Feature Store Statistics:")
stats_df

## 8. Feature Distribution Analysis

In [ ]:
sample_df = fs.read_feature_view(registered_fv).limit(50000).to_pandas()

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for i, col_name in enumerate(FEATURE_COLUMNS[:20]):
    ax = axes[i]
    if col_name in sample_df.columns:
        sample_df[col_name].hist(bins=50, ax=ax)
    ax.set_title(col_name, fontsize=9)
    ax.tick_params(labelsize=7)

plt.suptitle('Feature Distributions from Feature Store', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Temporal Train/Validation Split

In [ ]:
periods_df = session.sql(f"""
    SELECT DISTINCT PERIOD_END_DATE 
    FROM {SOURCE_TABLE}
    WHERE IS_ACTIVE = TRUE
    ORDER BY PERIOD_END_DATE
""").to_pandas()

num_periods = len(periods_df)
split_idx = int(num_periods * 0.8)
split_date = str(periods_df.iloc[split_idx]['PERIOD_END_DATE'])

print(f"Temporal split date: {split_date}")
print(f"Training data: periods before {split_date}")
print(f"Validation data: periods on/after {split_date}")

## 10. Retrieve Features for Training

Use point-in-time feature retrieval to avoid data leakage.

In [ ]:
features_snowpark = fs.read_feature_view(registered_fv)

train_df = features_snowpark.filter(col("PERIOD_END_DATE") < split_date).to_pandas()
val_df = features_snowpark.filter(col("PERIOD_END_DATE") >= split_date).to_pandas()

print(f"Training samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"\nTraining anomaly rate: {100*train_df['IS_ANOMALY_LABEL'].mean():.2f}%")
print(f"Validation anomaly rate: {100*val_df['IS_ANOMALY_LABEL'].mean():.2f}%")

## 11. Prepare Scaled Features

In [ ]:
X_train = train_df[FEATURE_COLUMNS].fillna(0).replace([np.inf, -np.inf], 0)
y_train = train_df['IS_ANOMALY_LABEL']

X_val = val_df[FEATURE_COLUMNS].fillna(0).replace([np.inf, -np.inf], 0)
y_val = val_df['IS_ANOMALY_LABEL']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_val shape: {X_val_scaled.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nValidation class distribution:")
print(y_val.value_counts())

## 12. Feature Correlation Matrix

In [ ]:
corr_matrix = train_df[FEATURE_COLUMNS].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 13. Save Artifacts for Training

In [ ]:
import pickle

artifacts = {
    'X_train_scaled': X_train_scaled,
    'y_train': y_train,
    'X_val_scaled': X_val_scaled,
    'y_val': y_val,
    'scaler': scaler,
    'feature_columns': FEATURE_COLUMNS,
    'split_date': split_date,
    'train_df': train_df,
    'val_df': val_df,
    'feature_view_name': registered_fv.name,
    'feature_view_version': 'v1'
}

with open('training_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("Artifacts saved to training_artifacts.pkl")
print(f"\nFeature Store Summary:")
print(f"  Feature View: {DATABASE}.{SCHEMA}.{registered_fv.name}$V1")
print(f"  Entity: {entity.name}")
print(f"  Features: {len(FEATURE_COLUMNS)}")
print(f"\nProceed to 03_training_experiments.ipynb")